# Demo — Stewardship on the Ambiguous Band

Identity resolution auto-merges the obvious duplicates (`score >= 0.85`, exact email and normalized
name) and rejects the obvious non-matches (`score < 0.66`) — but the **ambiguous band** in between
(*pairs that share a name and an email that differs only by mail provider or a trailing digit*)
needs human judgment. This walkthrough shows the match bands, compares an `auto-merge` pair against
two `ambiguous` pairs (a true duplicate vs. a homonym non-match), and demonstrates applying steward
adjudications with `apply_adjudications(..., survivor="lowest_id")`. The exercise asks you to
adjudicate all **eight** ambiguous pairs (`0..7`) and document your survivorship rule.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend, apply_adjudications

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

m = gc.resolve_customers()
print("auto-merge:", int((m.pairs.band == "auto-merge").sum()),
      "| ambiguous (review queue):", len(m.ambiguous),
      "| rejected:", int((m.pairs.band == "reject").sum()))

## Compare `auto-merge` vs. `ambiguous` candidate pairs
Pairs scoring `>= 0.85` share an exact email and normalized name, so the pipeline merges them
automatically. Pairs in the `0.66–0.84` band (*pairs that share a name and an email that differs
only by mail provider or a trailing digit*) land in `m.ambiguous` for steward review. Notice that
`m.ambiguous` includes `phone`, `country`, and `created_a`/`created_b` (`created_ts`) so you can
distinguish a re-registered customer from a homonym (you can also inspect full rows with `gc.query(...)`):

In [ ]:
print("--- Auto-merged pairs (score >= 0.85) ---")
display(m.pairs[m.pairs.band == "auto-merge"][["id_a", "name_a", "email_a", "id_b", "name_b", "email_b", "score", "band"]])

print("\n--- Contrast two ambiguous pairs: row 0 (same person) vs. row 6 (homonym non-match) ---")
m.ambiguous.iloc[[0, 6]][["id_a", "name_a", "email_a", "phone_a", "country_a", "created_a",
                          "id_b", "name_b", "email_b", "phone_b", "country_b", "created_b", "score"]]

## Adjudicate a true duplicate vs. a homonym and inspect survivorship
Row `0` (`Esme Reyes`, `45 ↔ 307`, `score == 0.80`) has the same name, email local-part, phone
(`+1058040488`), country (`CA`), and registration date (`2017-08-20`), with only a swapped email
domain (`trailmail.co` → `outlook.com`) — strong signal that one person registered twice. By
contrast, row `6` (`Zane Saleh`, `1 ↔ 309`, `score == 0.75`) has a different phone, country (`IN`
vs `US`), and registration date (`2024-12-12` vs `2022-03-14`) — two different people who happen to
share a name. Pass `{0: "accept", 6: "reject"}` and `survivor="lowest_id"` to `apply_adjudications()`
to see how the survivorship rule (`lowest customer_id wins`) selects the golden record:

In [ ]:
sample_outcome = apply_adjudications(m, decisions={0: "accept", 6: "reject"}, survivor="lowest_id")
print("Accepted merges:", sample_outcome["accepted_merges"],
      "| Rejected:", sample_outcome["rejected"],
      "of", sample_outcome["ambiguous_pairs"])
sample_outcome["merges"]

**Why human judgment matters:** accepting a true duplicate consolidates scattered loyalty and
marketing profiles into a single golden record (`golden_id`), whereas falsely merging a homonym pair
(two different people who share a name and email base across different countries/phones) would leak
one customer's PII and consent choices to another.

**Takeaway / your turn:** in the exercise you'll inspect and adjudicate all **eight** ambiguous pairs
(`0..7`), pass your survivorship rule into `apply_adjudications(m, decisions, survivor=...)`, and
write the per-pair stewardship decision log and survivorship rule.